# ⚽💰 Football Finance Intelligence
## Module 5 — Focus Maroc 🇲🇦 : la valeur économique des Lions de l'Atlas

**Idée :** appliquer les outils des modules précédents (valorisation, rendement, étude d'événement, comparaison) aux joueurs marocains et à l'équipe nationale.

### Questions étudiées
1. **Le vivier :** combien de joueurs marocains, dans quels championnats et quels clubs ?
2. **La diaspora :** quelle part des internationaux marocains est née à l'étranger ?
3. **L'évolution de la valeur** de la sélection de 2015 à 2026.
4. **L'effet Mondial 2022 :** la demi-finale historique a-t-elle fait monter la valeur des joueurs ?
5. **La comparaison africaine :** le Maroc face aux autres grandes nations du continent.

In [1]:
!pip install duckdb -q
!curl -sLO https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

con = duckdb.connect("transfermarkt-datasets.duckdb", read_only=True)
def q(sql):
    return con.sql(sql).df()

def chercher(table, motif):
    """Cherche un mot dans toutes les colonnes d'une table."""
    df = q(f"SELECT * FROM {table}")
    masque = df.astype(str).apply(lambda col: col.str.contains(motif, case=False, na=False)).any(axis=1)
    return df[masque]

# 1. La sélection marocaine dans national_teams
print("=== Colonnes de national_teams ===")
print(q("DESCRIBE national_teams")["column_name"].tolist())
display(chercher("national_teams", "Morocc"))

# 2. Le Mondial et la CAN dans competitions
print("\n=== Colonnes de competitions ===")
print(q("DESCRIBE competitions")["column_name"].tolist())
display(chercher("competitions", "World Cup|Africa|AFCON"))

# 3. Combien de joueurs marocains ?
display(q("""
SELECT COUNT(*) AS joueurs_marocains,
       SUM(CASE WHEN TRY_CAST(last_season AS INT) >= 2025 THEN 1 ELSE 0 END) AS encore_actifs
FROM players
WHERE country_of_citizenship = 'Morocco'
"""))

=== Colonnes de national_teams ===
['national_team_id', 'name', 'team_code', 'country_id', 'country_name', 'country_code', 'confederation', 'team_image_url', 'squad_size', 'average_age', 'foreigners_number', 'foreigners_percentage', 'total_market_value', 'coach_name', 'fifa_ranking', 'last_season', 'url']


,national_team_id,name,team_code,country_id,country_name,country_code,confederation,team_image_url,squad_size,average_age,foreigners_number,foreigners_percentage,total_market_value,coach_name,fifa_ranking,last_season,url
78,3575,Morocco,morocco,107,Morocco,MAR1,CAF,https://tmssl.akamaized.net//images/wappen/hom...,26,26.700001,24,92.300003,447700012,None,7,2025,https://www.transfermarkt.co.uk/morocco/starts...



=== Colonnes de competitions ===
['competition_id', 'competition_code', 'name', 'sub_type', 'type', 'country_id', 'country_name', 'domestic_league_code', 'confederation', 'total_clubs', 'url']


,competition_id,competition_code,name,sub_type,type,country_id,country_name,domestic_league_code,confederation,total_clubs,url
2,AFCN,africa-cup-of-nations,africa-cup-of-nations,africa_cup_of_nations,national_team_competition,-1,None,None,afrika,<NA>,https://www.transfermarkt.co.uk/africa-cup-of-...


,joueurs_marocains,encore_actifs
0,379,186.0


## 1. Exploration
### La sélection marocaine dans la base
| Indicateur | Valeur |
|---|---|
| Joueurs dans l'effectif | 26 |
| Âge moyen | 26,7 ans |
| Joueurs évoluant à l'étranger | 24 sur 26 (92 %) |
| Valeur totale de l'effectif | 447,7 M€ |
| Classement FIFA (selon la base) | 7ᵉ |

La sélection est quasi entièrement composée de joueurs évoluant **hors du Maroc**.

### Le vivier
La base recense **379 joueurs de nationalité marocaine**, dont **186 encore actifs** (dernière saison ≥ 2025).

*Note : la CAN est référencée sous le code `AFCN`. Les noms de compétitions utilisent des tirets (« africa-cup-of-nations »), ce qui explique que la recherche « World Cup » n'ait pas trouvé le Mondial.*

## 2. Le vivier marocain : où jouent les joueurs marocains ?
**Périmètre :** joueurs de nationalité marocaine encore actifs. On distingue les **sélectionnés** (rattachés à l'équipe nationale dans la base) des autres.

**Treemap :** chaque rectangle représente un championnat, puis un club, puis un joueur. Sa surface est proportionnelle à la **valeur marchande**.

## 3. La diaspora : où sont nés les Lions de l'Atlas ?
On compare le **pays de naissance** des joueurs sélectionnés et de l'ensemble du vivier.

In [2]:
# --- 1. Retrouver le Mondial ---
display(chercher("competitions", "world"))

# --- 2. Le vivier marocain actif ---
maroc = q("""
SELECT p.player_id, p.name, p.position,
       p.country_of_birth AS pays_naissance,
       p.current_club_name AS club,
       COALESCE(c.name, 'hors base') AS championnat,
       p.market_value_in_eur / 1e6 AS valeur_M,
       DATE_DIFF('day', CAST(p.date_of_birth AS DATE), DATE '2026-10-27') / 365.25 AS age,
       COALESCE(p.international_caps, 0) AS selections,
       CASE WHEN TRY_CAST(p.current_national_team_id AS DOUBLE) = 3575
            THEN 'Sélectionné' ELSE 'Non sélectionné' END AS statut
FROM players p
LEFT JOIN competitions c ON p.current_club_domestic_competition_id = c.competition_id
WHERE p.country_of_citizenship = 'Morocco'
  AND TRY_CAST(p.last_season AS INT) >= 2025
""")
maroc["championnat"] = maroc["championnat"].str.replace("-", " ").str.title()
maroc["club"] = maroc["club"].fillna("Club inconnu")
maroc["valeur_M"] = maroc["valeur_M"].fillna(0)

print(f"Joueurs marocains actifs : {len(maroc)}")
print(f"dont sélectionnés        : {(maroc['statut'] == 'Sélectionné').sum()}")
print(f"Valeur totale du vivier  : {maroc['valeur_M'].sum():,.0f} M€")

# Treemap : championnat → club → joueur
fig = px.treemap(maroc[maroc["valeur_M"] > 0],
                 path=[px.Constant("Joueurs marocains"), "championnat", "club", "name"],
                 values="valeur_M", color="statut",
                 color_discrete_map={"Sélectionné": "#C1272D", "Non sélectionné": "#006233",
                                     "(?)": "#DDDDDD"},
                 title="Où se trouve la valeur des joueurs marocains ? (taille = valeur marchande)")
fig.update_traces(textinfo="label+value")
fig.update_layout(height=700)
fig.show()

# Tableau par championnat
par_champ = (maroc.groupby("championnat")
                  .agg(nb_joueurs=("name", "count"),
                       selectionnes=("statut", lambda s: (s == "Sélectionné").sum()),
                       valeur_totale_M=("valeur_M", "sum"))
                  .sort_values("valeur_totale_M", ascending=False))
display(par_champ.head(12).round(1))

# --- 3. La diaspora ---
maroc["ne_au_maroc"] = np.where(maroc["pays_naissance"] == "Morocco", "Né au Maroc", "Né à l'étranger")
print("\nPart des joueurs nés à l'étranger :")
print((maroc.groupby("statut")["ne_au_maroc"]
            .apply(lambda s: (s == "Né à l'étranger").mean() * 100)).round(1).to_string())

pays = (maroc.groupby(["statut", "pays_naissance"]).size()
             .reset_index(name="nb_joueurs")
             .sort_values("nb_joueurs", ascending=False))
fig = px.bar(pays, x="pays_naissance", y="nb_joueurs", color="statut", barmode="group",
             color_discrete_map={"Sélectionné": "#C1272D", "Non sélectionné": "#006233"},
             title="Pays de naissance des joueurs marocains",
             labels={"pays_naissance": "Pays de naissance", "nb_joueurs": "Nombre de joueurs",
                     "statut": ""})
fig.show()

display(maroc[maroc["statut"] == "Sélectionné"]
        .sort_values("valeur_M", ascending=False)
        [["name", "position", "club", "pays_naissance", "age", "selections", "valeur_M"]]
        .round(1))

,competition_id,competition_code,name,sub_type,type,country_id,country_name,domestic_league_code,confederation,total_clubs,url
26,FIWC,world-cup,world-cup,world_cup,national_team_competition,-1,None,None,fifa,<NA>,https://www.transfermarkt.co.uk/world-cup/star...


Joueurs marocains actifs : 186
dont sélectionnés        : 43
Valeur totale du vivier  : 904 M€


,nb_joueurs,selectionnes,valeur_totale_M
championnat,,,
Ligue 1,26,0,274.9
Laliga,16,0,138.2
Eredivisie,20,0,106.5
Premier League,7,0,92.0
Bundesliga,3,0,65.0
Hors Base,43,43,47.8
Serie A,9,0,46.8
Jupiler Pro League,21,0,41.8
Super Lig,4,0,32.3



Part des joueurs nés à l'étranger :
statut
Non sélectionné    74.1
Sélectionné        11.6


,name,position,club,pays_naissance,age,selections,valeur_M
163,Soufiane Rahimi,Attack,Club inconnu,Morocco,30.4,39,6.0
154,Walid Azaro,Attack,Club inconnu,Morocco,31.4,16,3.0
151,Mounir Chouiar,Attack,Club inconnu,France,27.8,2,2.8
148,Tarik Tissoudali,Attack,Club inconnu,Netherlands,33.6,18,2.5
176,Amine Zouhzouh,Midfield,Club inconnu,Morocco,26.2,7,1.8
160,El Mehdi Benabid,Goalkeeper,Club inconnu,Morocco,28.8,5,1.5
161,Mohamed Rabie Hrimat,Midfield,Club inconnu,Morocco,32.2,15,1.5
157,Anas Bach,Midfield,Club inconnu,Morocco,28.7,12,1.5
159,Soufiane Bouftini,Defender,Club inconnu,Morocco,32.5,17,1.3
165,Ayoub Khairi,Midfield,Club inconnu,Morocco,26.7,4,1.2


### Lecture
**⚠️ Correction de méthode :** les 43 joueurs marqués « sélectionnés » ont tous un club inconnu, et Achraf Hakimi (102 sélections) n'en fait pas partie. La colonne `current_national_team_id` ne désigne donc pas les joueurs appelés en sélection : elle semble marquer les joueurs dont le club n'est pas suivi par la base (championnat marocain notamment). **Les internationaux seront redéfinis comme les joueurs ayant au moins une sélection.**

**Le vivier : 186 joueurs actifs pour 904 M€**
| Championnat | Joueurs | Valeur totale | Joueurs les plus chers |
|---|---|---|---|
| Ligue 1 | 26 | 275 M€ | Hakimi (80), Bouaddi (50), Igamane (18) |
| LaLiga | 16 | 138 M€ | Ezzalzouli (40), Brahim Díaz (35) |
| Eredivisie | 20 | 107 M€ | Saibari (40) |
| Premier League | 7 | 92 M€ | Talbi (25), Adli (20), Mazraoui (18) |
| Bundesliga | 3 | 65 M€ | El Khannouss (35), Ben Seghir (20) |
| Serie A | 9 | 47 M€ | El Aynaoui (23) |
| Belgique | 21 | 42 M€ | El Ouahdi (17) |

- **La Ligue 1 est le premier marché** des joueurs marocains (30 % de la valeur du vivier).
- **La Belgique et les Pays-Bas** accueillent de nombreux joueurs (41), souvent en début de carrière : des championnats tremplins.
- **La Premier League** compte peu de Marocains (7), mais à forte valeur (environ 13 M€ en moyenne).
- 43 joueurs évoluent dans des championnats non couverts (dont le championnat marocain), pour environ 48 M€.

**La diaspora :** seuls **75 joueurs sur 186 (40 %)** sont nés au Maroc. Les autres sont nés en **France** (34), aux **Pays-Bas** (26), en **Espagne** (19) et en **Belgique** (17), les principaux pays de la diaspora marocaine.

### Correction : les internationaux marocains
- **Vérification :** tableau croisé entre l'ancien indicateur « sélectionné » et l'absence de club.
- **Nouvelle définition :** un international est un joueur ayant **au moins une sélection**.
- **Effectif type :** les **26 internationaux les plus chers**, l'équivalent d'une liste pour une grande compétition.

In [4]:
# --- 1. Preuve : l'ancien indicateur = club inconnu ---
print("Ancien indicateur vs club inconnu :")
display(pd.crosstab(maroc["statut"], maroc["club"] == "Club inconnu",
                    colnames=["Club inconnu ?"]))

# --- 2. Nouvelle définition ---
maroc["international"] = np.where(maroc["selections"] >= 1,
                                  "International", "Jamais sélectionné")
print(maroc["international"].value_counts().to_string())

print("\n% de joueurs nés à l'étranger :")
print((maroc.groupby("international")["ne_au_maroc"]
            .apply(lambda s: (s == "Né à l'étranger").mean() * 100)).round(1).to_string())

pays = (maroc.groupby(["international", "pays_naissance"]).size()
             .reset_index(name="nb_joueurs")
             .sort_values("nb_joueurs", ascending=False))
fig = px.bar(pays, x="pays_naissance", y="nb_joueurs", color="international", barmode="group",
             color_discrete_map={"International": "#C1272D", "Jamais sélectionné": "#006233"},
             title="Pays de naissance des joueurs marocains : internationaux vs autres",
             labels={"pays_naissance": "Pays de naissance", "nb_joueurs": "Nombre de joueurs",
                     "international": ""})
fig.show()

# --- 3. L'effectif type : les 26 internationaux les plus chers ---
top26 = maroc[maroc["selections"] >= 1].nlargest(26, "valeur_M")
print(f"\nEffectif type (26 internationaux les plus chers) : {top26['valeur_M'].sum():,.0f} M€")
print(f"Âge moyen : {top26['age'].mean():.1f} ans")
nb_etranger = (top26["ne_au_maroc"] == "Né à l'étranger").sum()
print(f"Nés à l'étranger : {nb_etranger} sur 26")
fig = px.bar(top26.sort_values("valeur_M"), x="valeur_M", y="name", orientation="h",
             color="ne_au_maroc",
             color_discrete_map={"Né au Maroc": "#006233", "Né à l'étranger": "#C1272D"},
             hover_data={"club": True, "pays_naissance": True, "age": ":.1f", "selections": True},
             title="L'effectif type du Maroc : les 26 internationaux les plus chers",
             labels={"valeur_M": "Valeur marchande (M€)", "name": "", "ne_au_maroc": ""})
fig.update_layout(height=750)
fig.show()

display(top26[["name", "position", "club", "championnat", "pays_naissance",
               "age", "selections", "valeur_M"]].round(1))

Ancien indicateur vs club inconnu :


Club inconnu ?,False,True
statut,,
Non sélectionné,143,0
Sélectionné,0,43


international
International         154
Jamais sélectionné     32

% de joueurs nés à l'étranger :
international
International         55.8
Jamais sélectionné    78.1



Effectif type (26 internationaux les plus chers) : 609 M€
Âge moyen : 24.7 ans
Nés à l'étranger : 20 sur 26


,name,position,club,championnat,pays_naissance,age,selections,valeur_M
28,Achraf Hakimi,Defender,Paris Saint-Germain,Ligue 1,Spain,28.0,102,80.0
121,Ayyoub Bouaddi,Midfield,LOSC Lille,Ligue 1,France,19.1,8,50.0
70,Ismael Saibari,Midfield,PSV Eindhoven,Eredivisie,Spain,25.7,35,40.0
72,Abde Ezzalzouli,Attack,Real Betis Balompié,Laliga,Morocco,24.9,37,40.0
17,Brahim Díaz,Attack,Real Madrid,Laliga,Spain,27.2,32,35.0
63,Bilal El Khannouss,Midfield,VfB Stuttgart,Bundesliga,Belgium,22.5,43,35.0
77,Chemsdine Talbi,Attack,Sunderland AFC,Premier League,Belgium,21.5,10,25.0
99,Neil El Aynaoui,Midfield,Associazione Sportiva Roma,Serie A,France,25.3,22,23.0
125,Samir El Mourabet,Midfield,RC Strasbourg Alsace,Ligue 1,France,21.1,9,22.0
46,Amine Adli,Attack,AFC Bournemouth,Premier League,France,26.5,17,20.0


### Lecture
**Vérification :** les 43 joueurs marqués « sélectionnés » sont exactement les 43 joueurs sans club connu. L'ancien indicateur ne mesurait donc pas la sélection, et la correction est justifiée.

**Un vivier très international :** 154 joueurs sur 186 (83 %) comptent au moins une sélection.

**La diaspora au cœur de la sélection :** 56 % des internationaux sont nés à l'étranger, et **20 sur 26 (77 %)** dans l'effectif type.
| Pays de naissance | Effectif type |
|---|---|
| France | 7 |
| Espagne | 6 (dont Achraf Hakimi) |
| Pays-Bas | 4 |
| Belgique | 3 |
| Maroc | 6 |

💼 **Lecture financière :** la formation de la plupart de ces joueurs a été financée par des clubs européens. Le Maroc bénéficie de leur talent sans en avoir supporté le coût : une stratégie de **sourcing** efficace. Parmi les joueurs nés au Maroc, En-Nesyri et Aguerd sont issus de l'Académie Mohammed VI.

**L'effectif type : 609 M€, 24,7 ans de moyenne**
- Un noyau très jeune (Bouaddi, 19 ans et déjà 50 M€ ; Ait Boudlal, Yassine, El Mourabet, Talbi, Ben Seghir) : un **portefeuille d'actifs en phase d'appréciation** (Module 2).
- **Risque de concentration :** Achraf Hakimi représente à lui seul 13 % de la valeur totale.

*Limite : l'effectif type ne contient aucun gardien (valeurs plus faibles), alors qu'une liste réelle en compte trois. C'est l'une des raisons pour lesquelles il dépasse la valeur de la sélection indiquée par la base (447,7 M€).*

## 4. La valeur des Lions de l'Atlas dans le temps, face aux grandes nations africaines
**Méthode :** pour chaque année et chaque pays, on additionne la valeur des **26 internationaux les plus chers** (dernière estimation de l'année). On obtient la valeur de l'« effectif type » année après année, l'équivalent d'un cours de bourse de la sélection.

**Pays comparés :** Maroc, Sénégal, Égypte, Côte d'Ivoire, Nigeria, Algérie.

**Événements marqués :** Mondial 2022 (demi-finale du Maroc), CAN 2023 (jouée début 2024), CAN 2025 organisée au Maroc.

*Précautions : les années anciennes sont moins bien couvertes (biais de survie, Module 1). Le statut d'international est déterminé à partir du nombre de sélections actuel. L'année 2026 est partielle (estimations jusqu'en juin).*

In [5]:
afrique = q("""
WITH joueurs AS (
    SELECT player_id,
           CASE WHEN country_of_citizenship = 'Morocco' THEN 'Maroc'
                WHEN country_of_citizenship = 'Senegal' THEN 'Sénégal'
                WHEN country_of_citizenship = 'Egypt'   THEN 'Égypte'
                WHEN country_of_citizenship ILIKE '%ivoire%' THEN 'Côte d''Ivoire'
                WHEN country_of_citizenship = 'Nigeria' THEN 'Nigeria'
                WHEN country_of_citizenship = 'Algeria' THEN 'Algérie'
           END AS pays
    FROM players
    WHERE COALESCE(international_caps, 0) >= 1
),
annuel AS (
    SELECT player_id, YEAR(date) AS annee,
           arg_max(market_value_in_eur, date) AS valeur
    FROM player_valuations
    WHERE market_value_in_eur > 0
    GROUP BY 1, 2
),
classe AS (
    SELECT j.pays, a.annee, a.valeur,
           ROW_NUMBER() OVER (PARTITION BY j.pays, a.annee ORDER BY a.valeur DESC) AS rang
    FROM joueurs j
    JOIN annuel a ON j.player_id = a.player_id
    WHERE j.pays IS NOT NULL
      AND a.annee BETWEEN 2012 AND 2026
)
SELECT pays, annee,
       SUM(valeur) / 1e6 AS valeur_top26_M,
       COUNT(*) AS nb_joueurs
FROM classe
WHERE rang <= 26
GROUP BY 1, 2
ORDER BY 1, 2
""")

# --- Graphique : l'évolution des 6 nations ---
couleurs = {"Maroc": "#C1272D", "Sénégal": "#00853F", "Égypte": "#000000",
            "Côte d'Ivoire": "#F77F00", "Nigeria": "#1E90FF", "Algérie": "#9B59B6"}
fig = px.line(afrique, x="annee", y="valeur_top26_M", color="pays", markers=True,
              color_discrete_map=couleurs,
              title="Valeur de l'effectif type (26 internationaux les plus chers), 2012-2026",
              labels={"annee": "Année", "valeur_top26_M": "Valeur (M€)", "pays": "Pays"})
fig.update_traces(line=dict(width=1.5), opacity=0.6)
fig.update_traces(selector=dict(name="Maroc"), line=dict(width=4), opacity=1)
for x, texte in [(2022, "Mondial 2022"), (2024, "CAN 2023"), (2025, "CAN 2025 au Maroc")]:
    fig.add_vline(x=x, line_dash="dot", line_color="grey",
                  annotation_text=texte, annotation_position="top left")
fig.update_layout(hovermode="x unified", height=550)
fig.show()

# --- Tableau : valeur par pays en 2018, 2022 et 2025 ---
pivot = afrique.pivot(index="pays", columns="annee", values="valeur_top26_M")
resume = pivot[[2018, 2022, 2025]].copy()
resume["TCAM 2018-2025"] = ((pivot[2025] / pivot[2018]) ** (1 / 7) - 1) * 100
display(resume.sort_values(2025, ascending=False).round(1))

# --- Le Maroc année par année ---
maroc_annuel = afrique[afrique["pays"] == "Maroc"].copy()
maroc_annuel["variation_%"] = maroc_annuel["valeur_top26_M"].pct_change() * 100
display(maroc_annuel[["annee", "valeur_top26_M", "variation_%", "nb_joueurs"]].round(1))

annee,2018,2022,2025,TCAM 2018-2025
pays,,,,
Maroc,238.6,366.6,552.0,12.7
Côte d'Ivoire,368.6,444.0,526.0,5.2
Sénégal,343.8,321.0,454.0,4.0
Nigeria,207.0,401.5,393.5,9.6
Algérie,260.4,298.5,303.7,2.2
Égypte,191.8,145.3,134.1,-5.0


,annee,valeur_top26_M,variation_%,nb_joueurs
30,2012,20.7,NaN,15
31,2013,23.8,14.8,15
32,2014,40.3,69.4,20
33,2015,46.6,15.4,25
34,2016,83.2,78.8,26
35,2017,110.2,32.3,26
36,2018,238.6,116.6,26
37,2019,314.5,31.8,26
38,2020,305.2,-3.0,26
39,2021,349.3,14.4,26


### Lecture
**Le Maroc est devenu la sélection africaine la plus valorisée.**
| Pays | 2018 | 2022 | 2025 | TCAM 2018-2025 |
|---|---|---|---|---|
| Maroc | 239 M€ | 367 M€ | **552 M€** | **+12,7 %** |
| Côte d'Ivoire | 369 M€ | 444 M€ | 526 M€ | +5,2 % |
| Sénégal | 344 M€ | 321 M€ | 454 M€ | +4,0 % |
| Nigeria | 207 M€ | 402 M€ | 394 M€ | +9,6 % |
| Algérie | 260 M€ | 299 M€ | 304 M€ | +2,2 % |
| Égypte | 192 M€ | 145 M€ | 134 M€ | −5,0 % |

- **4ᵉ en 2018, 1ᵉʳ depuis 2024**, avec la croissance la plus forte des six pays. En 2026, le Maroc (609 M€) et la Côte d'Ivoire sont presque à égalité.
- **Accélération après le Mondial 2022 :** +5 % entre 2021 et 2022, puis **+20 %** en 2023 et +17 % en 2024. De 2022 à 2025, l'effectif type progresse de **+50 %**, contre +41 % pour le Sénégal et +18 % pour la Côte d'Ivoire.
- **Le bond de 2018 (+117 %) n'est pas spécifique au Maroc :** toutes les nations progressent fortement cette année-là, en raison de l'inflation générale du marché (Module 1).
- **2020 :** léger recul (−3 %) lié au Covid.
- **L'Égypte** est la seule nation en déclin (−5 % par an).

💡 **Leçon de méthode :** l'effet d'un événement ne peut être mesuré qu'en le comparant à un **groupe de référence**. C'est l'objet de la section suivante.

## 5. Étude d'événement : l'effet du Mondial 2022 sur la valeur des joueurs
**Principe (issu de la finance de marché) :** le **rendement anormal** est le rendement d'un actif **au-delà** de ce qu'on attendait sans l'événement.

**Méthode :**
1. **Joueurs traités :** ceux qui ont joué au moins un match du Mondial 2022.
2. **Rendement :** valeur après le Mondial (dernière estimation entre le 19/12/2022 et le 30/06/2023) ÷ valeur avant (dernière estimation avant le 20/11/2022) − 1.
3. **Groupe de référence :** joueurs n'ayant pas joué le Mondial, valant au moins 1 M€ avant la compétition. Leur rendement médian est calculé **par tranche d'âge**.
4. **Rendement anormal** = rendement du joueur − rendement médian des joueurs non mondialistes du même âge.

On compare ensuite les nations : lesquelles ont le plus profité du Mondial ?

In [14]:
# Vérification : de quelles saisons viennent les feuilles de match du Mondial ?
display(q("""
SELECT g.season, COUNT(*) AS nb_lignes
FROM game_lineups gl JOIN games g ON gl.game_id = g.game_id
WHERE g.competition_id = 'FIWC'
GROUP BY 1
"""))

etude = q("""
WITH nations_mondial AS (
    SELECT DISTINCT nt.country_name AS pays
    FROM games g
    JOIN national_teams nt
      ON nt.national_team_id = g.home_club_id OR nt.national_team_id = g.away_club_id
    WHERE g.competition_id = 'FIWC'
      AND CAST(g.date AS DATE) BETWEEN DATE '2022-11-20' AND DATE '2022-12-18'
),
avant AS (
    SELECT player_id, arg_max(market_value_in_eur, date) AS v_avant
    FROM player_valuations
    WHERE date BETWEEN DATE '2022-05-01' AND DATE '2022-11-19'
    GROUP BY 1
),
apres AS (
    SELECT player_id, arg_max(market_value_in_eur, date) AS v_apres
    FROM player_valuations
    WHERE date BETWEEN DATE '2022-12-19' AND DATE '2023-06-30'
    GROUP BY 1
),
base AS (
    SELECT p.player_id, p.name, p.country_of_citizenship AS pays, p.position,
           DATE_DIFF('day', CAST(p.date_of_birth AS DATE), DATE '2022-11-20') / 365.25 AS age,
           av.v_avant / 1e6 AS v_avant_M,
           ap.v_apres / 1e6 AS v_apres_M,
           COALESCE(p.international_caps, 0) AS selections
    FROM players p
    JOIN avant av ON p.player_id = av.player_id
    JOIN apres ap ON p.player_id = ap.player_id
    WHERE p.date_of_birth IS NOT NULL
),
classe AS (
    SELECT player_id,
           ROW_NUMBER() OVER (PARTITION BY pays ORDER BY v_avant_M DESC) AS rang
    FROM base
    WHERE selections >= 1
      AND pays IN (SELECT pays FROM nations_mondial)
)
SELECT b.*,
       CASE WHEN c.rang <= 26 THEN 1 ELSE 0 END AS mondialiste
FROM base b
LEFT JOIN classe c ON b.player_id = c.player_id
""")

nb_nations = etude.loc[etude["mondialiste"] == 1, "pays"].nunique()
print(f"{len(etude):,} joueurs, dont {etude['mondialiste'].sum()} mondialistes "
      f"(proxy) répartis dans {nb_nations} nations")

,season,nb_lignes
0,2025,3693


21,657 joueurs, dont 806 mondialistes (proxy) répartis dans 31 nations


In [15]:
etude["rendement"] = etude["v_apres_M"] / etude["v_avant_M"] - 1
etude["tranche_age"] = pd.cut(etude["age"], bins=[0, 21, 24, 27, 30, 45],
                              labels=["21 et -", "22-24", "25-27", "28-30", "31 et +"])

reference = etude[(etude["mondialiste"] == 0) & (etude["v_avant_M"] >= 1)]
normal = reference.groupby("tranche_age", observed=True)["rendement"].median()
print("Rendement normal par âge (%) :")
print((normal * 100).round(1).to_string())

etude["rendement_normal"] = etude["tranche_age"].map(normal).astype(float)
etude["rendement_anormal"] = etude["rendement"] - etude["rendement_normal"]
mondialistes = etude[etude["mondialiste"] == 1].copy()
print(f"\nMondialistes analysés : {len(mondialistes)}")

Rendement normal par âge (%) :
tranche_age
21 et -     0.0
22-24       0.0
25-27       0.0
28-30     -10.0
31 et +   -20.0

Mondialistes analysés : 806


In [16]:
nations = (mondialistes.groupby("pays")
           .agg(nb_joueurs=("name", "count"),
                rendement_median=("rendement", "median"),
                rendement_anormal_median=("rendement_anormal", "median"))
           .reset_index())
nations = nations[nations["nb_joueurs"] >= 8].sort_values("rendement_anormal_median")
nations[["rendement_median", "rendement_anormal_median"]] *= 100
nations["couleur"] = np.where(nations["pays"] == "Morocco", "Maroc", "Autres nations")

fig = px.bar(nations, x="rendement_anormal_median", y="pays", orientation="h",
             color="couleur", color_discrete_map={"Maroc": "#C1272D", "Autres nations": "#AAAAAA"},
             hover_data={"nb_joueurs": True, "rendement_median": ":.1f", "couleur": False},
             title="Mondial 2022 : rendement anormal médian des joueurs, par nation",
             labels={"rendement_anormal_median": "Rendement anormal médian (%)",
                     "pays": "", "couleur": ""})
fig.add_vline(x=0, line_color="black")
fig.update_layout(height=750)
fig.show()

lions = mondialistes[mondialistes["pays"] == "Morocco"]
lions = lions.sort_values("rendement_anormal", ascending=False)
display(lions[["name", "position", "age", "v_avant_M", "v_apres_M",
               "rendement", "rendement_anormal"]].round(2))

v_avant = lions["v_avant_M"].sum()
v_apres = lions["v_apres_M"].sum()
print(f"Lions de l'Atlas : {v_avant:.0f} M€ avant → {v_apres:.0f} M€ après "
      f"({v_apres / v_avant - 1:+.1%})")

,name,position,age,v_avant_M,v_apres_M,rendement,rendement_anormal
156,Azzedine Ounahi,Midfield,22.59,3.5,12.0,2.43,2.43
3902,Anass Zaroury,Attack,22.03,4.0,12.0,2.00,2.00
11041,Sofyan Amrabat,Midfield,26.25,10.0,30.0,2.00,2.00
14776,Bilal El Khannouss,Midfield,18.53,4.5,13.0,1.89,1.89
7532,Abde Ezzalzouli,Attack,20.93,8.0,15.0,0.88,0.88
149,Youssef En-Nesyri,Attack,25.47,12.0,20.0,0.67,0.67
14937,Amine Adli,Attack,22.53,12.0,20.0,0.67,0.67
14672,Nayef Aguerd,Defender,26.64,25.0,35.0,0.40,0.40
20382,Oussama Idrissi,Attack,26.73,4.0,5.5,0.38,0.38
16627,Soufiane Rahimi,Attack,26.47,4.5,5.5,0.22,0.22


Lions de l'Atlas : 320 M€ avant → 391 M€ après (+22.2%)


### Lecture
**1. Le Maroc parmi les nations ayant le plus profité du Mondial.** Rendement anormal médian d'environ **+7 %**, dans le top 6 des 32 nations.
- **Japon** (+50 %) : ses victoires contre l'Allemagne et l'Espagne ont révélé ses joueurs au marché européen.
- **Croatie** (−9 %), pourtant 3ᵉ : un effectif âgé, dont la dépréciation n'est pas compensée par la performance.
- **Arabie saoudite** (−6 %) : malgré sa victoire contre l'Argentine, ses joueurs évoluent dans un championnat peu suivi par le marché européen.

**2. Les Lions de l'Atlas : 320 M€ → 391 M€ (+22 %) en 6 mois.**
| Joueur | Avant | Après | Rendement |
|---|---|---|---|
| Azzedine Ounahi | 3,5 M€ | 12 M€ | **+243 %** |
| Anass Zaroury | 4 M€ | 12 M€ | +200 % |
| Sofyan Amrabat | 10 M€ | 30 M€ | +200 % |
| Bilal El Khannouss | 4,5 M€ | 13 M€ | +189 % |
| Abde Ezzalzouli | 8 M€ | 15 M€ | +88 % |
| Youssef En-Nesyri | 12 M€ | 20 M€ | +67 % |
| Nayef Aguerd | 25 M€ | 35 M€ | +40 % |

**3. 💡 Le Mondial révèle les talents, il n'enrichit pas les stars.**
Achraf Hakimi (65 → 65 M€) et Hakim Ziyech n'ont pratiquement pas pris de valeur, malgré un excellent tournoi. À l'inverse, des joueurs peu exposés comme Ounahi ou Amrabat ont été fortement réévalués.
C'est l'**hypothèse d'efficience des marchés** : le prix d'un actif reflète l'information disponible. Le marché connaissait déjà la valeur des stars. Le Mondial a apporté une **information nouvelle** sur les joueurs moins exposés, ce qui a provoqué leur réévaluation.

*Yassine Bounou : rendement brut de −20 %, mais rendement anormal de 0 %. À 31 ans, la baisse normale est de −20 % : son Mondial a compensé l'effet de l'âge.*

⚠️ **Limites :** la base ne contient pas les listes du Mondial 2022. Le proxy (26 internationaux les plus chers) inclut des joueurs arrivés en sélection après 2022 (Brahim Díaz, Sofiane Diop) et des absents (Amine Harit, blessé). Les résultats sont donc des **approximations**.

---
# ✅ Conclusion du Module 5 — Focus Maroc 🇲🇦

### Les 5 messages clés
1. **Un vivier de 186 joueurs actifs pour 904 M€**, dont la Ligue 1 est le premier marché (30 % de la valeur). La Belgique et les Pays-Bas servent de championnats tremplins.
2. **Une sélection construite sur la diaspora :** 56 % des internationaux et 77 % de l'effectif type sont nés à l'étranger (France, Espagne, Pays-Bas, Belgique). Le Maroc bénéficie de joueurs formés et financés par des clubs européens.
3. **La sélection africaine la plus valorisée :** 4ᵉ en 2018, 1ʳᵉ depuis 2024, avec la croissance la plus forte du continent (+12,7 % par an de 2018 à 2025).
4. **Un effectif jeune, en phase d'appréciation :** 24,7 ans de moyenne pour l'effectif type, avec plusieurs joueurs de moins de 22 ans déjà valorisés entre 20 et 50 M€.
5. **L'effet Mondial 2022 :** +22 % de valeur en 6 mois pour les Lions de l'Atlas, porté par la révélation de joueurs peu exposés (Ounahi, Amrabat), ce qui illustre l'efficience des marchés.

### Démarche et rigueur
- **Correction d'un indicateur trompeur :** la colonne « sélection nationale » désignait en réalité les joueurs sans club suivi.
- **Recherche d'une source alternative :** les listes du Mondial 2022 étant absentes de la base, construction d'un proxy explicitement documenté.
- **Comparaison à un groupe de référence** (rendement anormal), pour isoler l'effet propre du Mondial.

### Limites
- Statut d'international basé sur le nombre de sélections actuel.
- Championnat marocain non couvert par la base.
- Proxy des listes du Mondial 2022.